In [43]:
#Defining the layer class. 
# For the forward bias it takes the input and passes it to the output.
# And the reverse  bias takes the output gradient and learning rate as input its updates the parameters 
# and gives input gradient as output which is taken as the input for next layer.

class Layer:
    def __init__(self):
        self.input = None
        self.output = None

    def forward(self, input):
        # TODO: return output
        pass

    def backward(self, output_gradient, learning_rate):
        # TODO: update parameters and return input gradient
        pass

In [44]:
#For the dense layer we have neurons connected to all other neurons in the next layer.
# The forward propagation is the y=w.x+b
# The backward propogation is weight_gradient= lr*output_gradient*xT
# bias_gradient= lr*output gradient

import numpy as np

class Dense(Layer):
    def __init__(self, input_size, output_size):
        self.weights = np.random.randn(output_size, input_size)
        self.bias = np.random.randn(output_size, 1)

    def forward(self, input):
        self.input = input
        return np.dot(self.weights, self.input) + self.bias

    def backward(self, output_gradient, learning_rate):
        weights_gradient = np.dot(output_gradient, self.input.T)
        input_gradient = np.dot(self.weights.T, output_gradient)
        self.weights -= learning_rate * weights_gradient
        self.bias -= learning_rate * output_gradient
        return input_gradient

import numpy as np

class Dense(Layer):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.weights = np.random.randn(output_size, input_size)
        self.bias = np.random.randn(output_size, 1)

    def forward(self, input):
        self.input = input
        self.output = np.dot(self.weights, self.input) + self.bias
        print(f"\n--- Forward: Dense Layer ---")
        print(f"Input:\n{self.input}")
        print(f"Weights:\n{self.weights}")
        print(f"Bias:\n{self.bias}")
        print(f"Output:\n{self.output}")
        return self.output

    def backward(self, output_gradient, learning_rate):
        weights_gradient = np.dot(output_gradient, self.input.T)
        input_gradient = np.dot(self.weights.T, output_gradient)

        print(f"\n--- Backward: Dense Layer ---")
        print(f"Output Gradient:\n{output_gradient}")
        print(f"Weights Gradient:\n{weights_gradient}")
        print(f"Input Gradient:\n{input_gradient}")
        print(f"Weights Before Update:\n{self.weights}")
        print(f"Bias Before Update:\n{self.bias}")

        self.weights -= learning_rate * weights_gradient
        self.bias -= learning_rate * output_gradient

        print(f"Weights After Update:\n{self.weights}")
        print(f"Bias After Update:\n{self.bias}")

        return input_gradient


In [45]:
import numpy as np

# Create a Dense layer: 2 inputs → 3 outputs
layer = Dense(2, 3)

# Random input (2x1 vector)
x = np.array([[1.0], [0.5]])

# Forward pass
output = layer.forward(x)
print("\nFinal Forward Output:\n", output)

# Dummy output gradient (same shape as output)
output_gradient = np.ones_like(output)

# Backward pass
input_gradient = layer.backward(output_gradient, learning_rate=0.1)
print("\nFinal Input Gradient (Backpropagated):\n", input_gradient)



Final Forward Output:
 [[0.65396866]
 [1.41288935]
 [2.06161772]]

Final Input Gradient (Backpropagated):
 [[1.35247922]
 [2.24181396]]


In [46]:
import numpy as np

def mse(y_true, y_pred):
    return np.mean(np.power(y_true - y_pred, 2))

def mse_prime(y_true, y_pred):
    return 2 * (y_pred - y_true) / np.size(y_true)

def binary_cross_entropy(y_true, y_pred):
    return np.mean(-y_true * np.log(y_pred) - (1 - y_true) * np.log(1 - y_pred))

def binary_cross_entropy_prime(y_true, y_pred):
    return ((1 - y_true) / (1 - y_pred) - y_true / y_pred) / np.size(y_true)

In [47]:
import numpy as np

class Activation(Layer):
    def __init__(self, activation, activation_prime):
        self.activation = activation
        self.activation_prime = activation_prime

    def forward(self, input):
        self.input = input
        return self.activation(self.input)

    def backward(self, output_gradient, learning_rate):
        return np.multiply(output_gradient, self.activation_prime(self.input))

In [48]:
import numpy as np

class Tanh(Activation):
    def __init__(self):
        def tanh(x):
            return np.tanh(x)

        def tanh_prime(x):
            return 1 - np.tanh(x) ** 2

        super().__init__(tanh, tanh_prime)

class Sigmoid(Activation):
    def __init__(self):
        def sigmoid(x):
            return 1 / (1 + np.exp(-x))

        def sigmoid_prime(x):
            s = sigmoid(x)
            return s * (1 - s)

        super().__init__(sigmoid, sigmoid_prime)

class Softmax(Layer):
    def forward(self, input):
        tmp = np.exp(input)
        self.output = tmp / np.sum(tmp)
        return self.output
    
    def backward(self, output_gradient, learning_rate):
        # This version is faster than the one presented in the video
        n = np.size(self.output)
        return np.dot((np.identity(n) - self.output.T) * self.output, output_gradient)
        # Original formula:
        # tmp = np.tile(self.output, n)
        # return np.dot(tmp * (np.identity(n) - np.transpose(tmp)), output_gradient)

In [49]:
import numpy as np

class Reshape(Layer):
    def __init__(self, input_shape, output_shape):
        self.input_shape = input_shape
        self.output_shape = output_shape

    def forward(self, input):
        return np.reshape(input, self.output_shape)

    def backward(self, output_gradient, learning_rate):
        return np.reshape(output_gradient, self.input_shape)    

In [50]:
def predict(network, input):
    output = input
    for layer in network:
        output = layer.forward(output)
    return output

def train(network, loss, loss_prime, x_train, y_train, epochs = 1000, learning_rate = 0.01, verbose = True):
    for e in range(epochs):
        error = 0
        for x, y in zip(x_train, y_train):
            # forward
            output = predict(network, x)

            # error
            error += loss(y, output)

            # backward
            grad = loss_prime(y, output)
            for layer in reversed(network):
                grad = layer.backward(grad, learning_rate)

        error /= len(x_train)
        if verbose:
            print(f"{e + 1}/{epochs}, error={error}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# XOR dataset
X = np.reshape([[0, 0], [0, 1], [1, 0], [1, 1]], (4, 2, 1))
Y = np.reshape([[0], [1], [1], [0]], (4, 1, 1))

# Define the network
network = [
    Dense(2, 3),
    Tanh(),
    Dense(3, 1),
    Tanh()
]

# Train the network
train(network, mse, mse_prime, X, Y, epochs=10000, learning_rate=0.1)

# Test the network and calculate accuracy
correct_predictions = 0
for x, y in zip(X, Y):
    output = predict(network, x)
    predicted = 1 if output[0, 0] >= 0.5 else 0  # Threshold at 0.5
    if predicted == y[0, 0]:
        correct_predictions += 1
    print(f"Input: {x.flatten()}, Predicted: {predicted}, True: {int(y[0, 0])}")

accuracy = (correct_predictions / len(X)) * 100
print(f"\nAccuracy: {accuracy:.2f}%")

# Decision boundary plot
points = []
for x in np.linspace(0, 1, 20):
    for y in np.linspace(0, 1, 20):
        z = predict(network, [[x], [y]])
        points.append([x, y, z[0, 0]])

points = np.array(points)

fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=points[:, 2], cmap="winter")
plt.show()

In [65]:
import numpy as np

# Activation Layer for Tanh
class TanhActivation:
    def forward(self, input):
        self.input = input
        return np.tanh(input)

    def backward(self, output_gradient):
        return output_gradient * (1 - np.tanh(self.input) ** 2)

# Dense Layer
class Dense:
    def __init__(self, input_size, output_size):
        self.weights = np.random.randn(output_size, input_size) / np.sqrt(input_size)  # Xavier Initialization
        self.biases = np.random.randn(output_size, 1) / np.sqrt(input_size)

    def forward(self, input):
        self.input = input
        return np.dot(self.weights, input) + self.biases
    def backward(self, output_gradient, learning_rate):
        weights_gradient = np.dot(output_gradient, self.input.T)
        input_gradient = np.dot(self.weights.T, output_gradient)
        self.weights -= learning_rate * weights_gradient
        self.biases -= learning_rate * output_gradient
        return input_gradient

# Mean Squared Error Loss
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mse_prime(y_true, y_pred):
    return 2 * (y_pred - y_true) / y_true.size

# XOR Dataset
inputs = np.array([
    [[0.0], [0.0]],
    [[0.0], [1.0]],
    [[1.0], [0.0]],
    [[1.0], [1.0]]
])
outputs = np.array([
    [[0.0]],
    [[1.0]],
    [[1.0]],
    [[0.0]]
])

# Create Layers
layer1 = Dense(2, 3)  # First layer: 2 inputs → 3 outputs
activation1 = TanhActivation()
layer2 = Dense(3, 1)  # Second layer: 3 inputs → 1 output
activation2 = TanhActivation()

# Training Parameters
epochs = 10000
learning_rate = 0.1

# Train the Model
for epoch in range(epochs):
    total_error = 0.0
    for x, y in zip(inputs, outputs):
        # Forward pass
        hidden_output = activation1.forward(layer1.forward(x))
        output = activation2.forward(layer2.forward(hidden_output))

        # Compute erro
        total_error += mse(y,output)

        # Backward pass
        output_gradient = mse_prime(y, output)
        hidden_gradient = layer2.backward(activation2.backward(output_gradient), learning_rate)
        layer1.backward(activation1.backward(hidden_gradient), learning_rate)

    # Print error every 10 epochs
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Total Error: {total_error / len(inputs)}")

# Test the Model
print("\nTesting the model:")
correct_predictions = 0
for x, y in zip(inputs, outputs):
    hidden_output = activation1.forward(layer1.forward(x))
    output = activation2.forward(layer2.forward(hidden_output))
    predicted = 1.0 if output[0, 0] >= 0.5 else 0.0
    if predicted == y[0, 0]:
        correct_predictions += 1
    print(f"Input: {x.flatten()}, Predicted: {predicted}, True: {y[0, 0]}")

# Calculate and print accuracy
accuracy = (correct_predictions / len(inputs)) * 100
print(f"\nAccuracy: {accuracy}%")

Epoch 0, Total Error: 0.382978613847055
Epoch 10, Total Error: 0.3099012307988522
Epoch 20, Total Error: 0.28273828495555975
Epoch 30, Total Error: 0.25508537827260847
Epoch 40, Total Error: 0.23105910596477391
Epoch 50, Total Error: 0.21286040928725447
Epoch 60, Total Error: 0.19683264510707354
Epoch 70, Total Error: 0.1738401559199878
Epoch 80, Total Error: 0.128136111637158
Epoch 90, Total Error: 0.06497796209623626
Epoch 100, Total Error: 0.030370682589489318
Epoch 110, Total Error: 0.01759927665713372
Epoch 120, Total Error: 0.011901951374905927
Epoch 130, Total Error: 0.00882290161530145
Epoch 140, Total Error: 0.006935014579083362
Epoch 150, Total Error: 0.005674177417615574
Epoch 160, Total Error: 0.004779210900820776
Epoch 170, Total Error: 0.004114449605649575
Epoch 180, Total Error: 0.0036030847483715153
Epoch 190, Total Error: 0.003198636867344307
Epoch 200, Total Error: 0.002871453247305837
Epoch 210, Total Error: 0.0026017864701022436
Epoch 220, Total Error: 0.00237600996

In [61]:
import numpy as np

# --- Activation Layer for Tanh ---
class TanhActivation:
    def forward(self, input):
        self.input = input
        return np.tanh(input)

    def backward(self, output_gradient):
        return output_gradient * (1 - np.tanh(self.input) ** 2)

# --- Dense Layer ---
class Dense:
    def __init__(self, input_size, output_size):
        self.weights = np.random.randn(output_size, input_size) / np.sqrt(input_size)
        self.biases = np.random.randn(output_size, 1) / np.sqrt(input_size)

    def forward(self, input):
        self.input = input
        return np.dot(self.weights, input) + self.biases

    def backward(self, output_gradient, learning_rate):
        weights_gradient = np.dot(output_gradient, self.input.T)
        input_gradient = np.dot(self.weights.T, output_gradient)
        self.weights -= learning_rate * weights_gradient
        self.biases -= learning_rate * output_gradient
        return input_gradient

# --- Mean Squared Error Loss ---
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mse_prime(y_true, y_pred):
    return 2 * (y_pred - y_true) / y_true.size

# --- XOR Dataset ---
inputs = np.array([
    [[0.0], [0.0]],
    [[0.0], [1.0]],
    [[1.0], [0.0]],
    [[1.0], [1.0]]
])
outputs = np.array([
    [[0.0]],
    [[1.0]],
    [[1.0]],
    [[0.0]]
])

# --- Create Layers ---
layer1 = Dense(2, 3)
activation1 = TanhActivation()
layer2 = Dense(3, 1)
activation2 = TanhActivation()

# --- Training Parameters ---
epochs = 10000
learning_rate = 0.1

# --- Train the Model ---
for epoch in range(epochs):
    total_error = 0.0
    for x, y in zip(inputs, outputs):
        # Forward pass
        z1 = layer1.forward(x)
        a1 = activation1.forward(z1)
        z2 = layer2.forward(a1)
        a2 = activation2.forward(z2)

        # Compute and accumulate loss
        total_error += mse(y, a2)

        # Backward pass
        grad = mse_prime(y, a2)
        grad = activation2.backward(grad)
        grad = layer2.backward(grad, learning_rate)
        grad = activation1.backward(grad)
        layer1.backward(grad, learning_rate)

    # Print error every 100 epochs
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, MSE: {total_error / len(inputs)}")

# --- Test the Model ---
print("\nTesting the model:")
correct_predictions = 0
for x, y in zip(inputs, outputs):
    a1 = activation1.forward(layer1.forward(x))
    output = activation2.forward(layer2.forward(a1))
    predicted = 1.0 if output[0, 0] >= 0.5 else 0.0
    if predicted == y[0, 0]:
        correct_predictions += 1
    print(f"Input: {x.flatten()}, Predicted: {predicted}, True: {y[0, 0]}")

# --- Accuracy ---
accuracy = (correct_predictions / len(inputs)) * 100
print(f"\nAccuracy: {accuracy}%")


Epoch 0, MSE: 0.29774612752542934
Epoch 100, MSE: 0.020547309962254328
Epoch 200, MSE: 0.002331335019640203
Epoch 300, MSE: 0.0011085339199107644
Epoch 400, MSE: 0.0007078424311950082
Epoch 500, MSE: 0.0005136779617461179
Epoch 600, MSE: 0.0004003956987760037
Epoch 700, MSE: 0.0003266561121727636
Epoch 800, MSE: 0.00027505104250662903
Epoch 900, MSE: 0.00023702712490159992
Epoch 1000, MSE: 0.00020790937646428321
Epoch 1100, MSE: 0.00018493451628055587
Epoch 1200, MSE: 0.0001663680407278876
Epoch 1300, MSE: 0.00015106805680267817
Epoch 1400, MSE: 0.00013825310729725744
Epoch 1500, MSE: 0.0001273709316141667
Epoch 1600, MSE: 0.00011802052250820867
Epoch 1700, MSE: 0.00010990387735340036
Epoch 1800, MSE: 0.00010279505943587908
Epoch 1900, MSE: 9.651974721273621e-05
Epoch 2000, MSE: 9.09413538563117e-05
Epoch 2100, MSE: 8.595138461674267e-05
Epoch 2200, MSE: 8.146259873019606e-05
Epoch 2300, MSE: 7.740407020488073e-05
Epoch 2400, MSE: 7.371756080118545e-05
Epoch 2500, MSE: 7.03548165972137